# Hadoop to Spark — Pivot Guide

Today's lab: feel **MapReduce friction** from prior classes, then solve the same ShopStream analytics with **PySpark**.

| | |
|---|---|
| **Scenario** | ShopStream nightly report — reviews + orders + sentiment |
| **Prerequisite** | MapReduce lab complete — [MAPREDUCE-STUDENT-GUIDE.md](../MAPREDUCE-STUDENT-GUIDE.md) |
| **Run from** | `hadoop-local-docker/spark/` |
| **Written guide** | [SPARK-STUDENT-GUIDE.md](./SPARK-STUDENT-GUIDE.md) |

**How to use:** Run cells **in order**. Hadoop cluster optional for Step 3 demo.

---
## Step 1 — The problem MapReduce makes hard

ShopStream needs **one report** with:

1. Top words in product reviews
2. Revenue by category from orders
3. Categories with the most negative review keywords

In MapReduce, that usually means **multiple YARN jobs** and **HDFS temp folders** between steps.

```mermaid
flowchart TB
  subgraph goal [One business report]
    G1[Word frequencies]
    G2[Revenue by category]
    G3[Category sentiment join]
  end

  subgraph mr [MapReduce — separate batch jobs]
    RAW[(HDFS raw)] --> JOB1[Job 1 WordCount]
    JOB1 --> TMP1[(HDFS /tmp/step1)]
    RAW --> JOB2[Job 2 Revenue agg]
    JOB2 --> TMP2[(HDFS /tmp/step2)]
    TMP1 --> JOB3[Job 3 Custom join]
    TMP2 --> JOB3
    JOB3 --> RPT[(HDFS report)]
  end

  G1 -.-> JOB1
  G2 -.-> JOB2
  G3 -.-> JOB3
```

---
## Step 2 — MapReduce bottlenecks you already hit

```mermaid
flowchart LR
  subgraph write [1 — Author code]
    MAP[mapper.py]
    RED[reducer.py]
  end

  subgraph deploy [2 — Deploy]
    CP[docker cp scripts]
    JAR[hadoop-streaming JAR]
  end

  subgraph run [3 — Execute]
    YARN[YARN allocates containers]
    DISK[(Shuffle spills to disk)]
    HDFS[(Write HDFS output)]
  end

  subgraph repeat [4 — Next question]
    AGAIN[Repeat for every step]
  end

  MAP --> CP --> YARN --> DISK --> HDFS --> AGAIN
  RED --> CP
  JAR --> YARN
```

| Symptom from MapReduce lab | Root cause |
|----------------------------|------------|
| `Output directory already exists` | Each job needs its own HDFS path |
| Minutes before `map 100%` | YARN startup + container allocation |
| `Reduce shuffle bytes` in logs | Sort/shuffle materialized to disk + network |
| Python 2.7 in container | Hadoop Streaming runtime lock-in |
| New scripts for each analysis | MapReduce programming model is narrow |

In [ ]:
import sys
from pathlib import Path

# Allow imports from spark/ when kernel cwd is hadoop-local-docker/
spark_dir = Path("spark" if Path("spark/spark_helpers.py").exists() else ".").resolve()
if str(spark_dir) not in sys.path:
    sys.path.insert(0, str(spark_dir))

from hadoop_pain_points import print_pain_summary

print_pain_summary()

---
## Step 3 — Optional: feel MapReduce job latency live

If your Docker Hadoop cluster is **running**, submit one wordcount job and note the friction: deploy scripts, wait for YARN, shuffle counters, read HDFS output.

If the cluster is **down**, read the printed message and continue — Spark local mode does not require Hadoop.

In [ ]:
import subprocess

def cluster_running():
    result = subprocess.run(
        "docker inspect --format='{{.State.Health.Status}}' namenode",
        shell=True, capture_output=True, text=True,
    )
    return result.stdout.strip() == "healthy"

if cluster_running():
    print("Cluster healthy — running one MapReduce wordcount for comparison...")
    sys.path.insert(0, str(Path("..").resolve()))
    from mapreduce.job_helpers import deploy_scripts, submit_streaming_job, read_hdfs_output

    deploy_scripts("../mapreduce/streaming", ["wordcount_mapper.py", "wordcount_reducer.py"])
    submit_streaming_job(
        "wordcount_mapper.py", "wordcount_reducer.py",
        "/shopstream/raw/reviews/product_reviews.txt",
        "/shopstream/processed/spark_pivot_mr_wordcount",
    )
    read_hdfs_output("/shopstream/processed/spark_pivot_mr_wordcount", top_n=5)
    print("\nNotice: deploy + YARN wait + HDFS cleanup required for ONE question.")
else:
    print("Hadoop cluster not running — skip live MR demo.")
    print("Start with: cd .. && docker compose up -d")

**Checkpoint:** One MapReduce answer required scripts, a YARN job, and an HDFS output path. The next steps do all three report questions in **one Spark session**.

---
## Step 4 — Spark architecture (the pivot)

```mermaid
flowchart TB
  subgraph driver [Driver — this notebook]
    PY[PySpark code]
    PLAN[Lazy DAG plan]
  end

  subgraph exec [Executors — local JVM threads]
    T1[Read CSV]
    T2[Transform]
    T3[Shuffle if needed]
    T4[Aggregate]
  end

  subgraph data [Storage]
    CSV[(ecommerce CSV files)]
    MEM[(Cached DataFrames)]
  end

  PY --> PLAN --> T1
  T1 --> CSV
  T1 --> T2 --> T3 --> T4
  T2 --> MEM
  T4 --> OUT[Results — show or write once]
```

| Concept | MapReduce | Spark |
|---------|-----------|-------|
| Program shape | mapper + reducer files | PySpark script / notebook |
| Execution trigger | Each `hadoop jar` submit | Actions like `.show()` / `.count()` |
| Multi-step pipeline | Many jobs | One app, many transforms |
| Monitor | YARN 8088 + History 19888 | Spark UI 4040 |

---
## Step 5 — Start PySpark (local mode)

Open http://localhost:4040 after running the next cell — Spark UI shows jobs and stages.

In [ ]:
from pyspark.sql import functions as F
from spark_helpers import create_spark, data_path

spark = create_spark("ShopStream-Pivot")
print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)
print("Spark UI: http://localhost:4040")

---
## Step 6 — Same word count — Spark vs MapReduce code size

```mermaid
flowchart LR
  subgraph hadoop [MapReduce streaming]
    H1[mapper.py] --> H2[sort]
    H2 --> H3[reducer.py]
    H3 --> H4[deploy + YARN job]
  end

  subgraph spark [PySpark]
    S1[read.text] --> S2[split explode]
    S2 --> S3[groupBy count]
  end
```

In [ ]:
reviews = spark.read.text(data_path("product_reviews.txt"))

top_words = (
    reviews.select(
        F.explode(F.split(F.lower(F.col("value")), r"\W+")).alias("word")
    )
    .where(F.col("word") != "")
    .groupBy("word")
    .count()
    .orderBy(F.desc("count"))
)

print("Top 10 words — PySpark")
top_words.show(10, truncate=False)

**Checkpoint:** Same business question as MapReduce wordcount — fewer lines, no deploy step, no HDFS output cleanup.

---
## Step 7 — Revenue by category (second report question)

In MapReduce this would be **another job**. In Spark it is the next cell.

In [ ]:
orders = (
    spark.read.option("header", True)
    .csv(data_path("orders.csv"))
    .withColumn("revenue", F.col("quantity").cast("int") * F.col("unit_price").cast("double"))
)

revenue_by_category = (
    orders.groupBy("category")
    .agg(F.round(F.sum("revenue"), 2).alias("total_revenue"), F.count("*").alias("order_lines"))
    .orderBy(F.desc("total_revenue"))
)

print("Revenue by category")
revenue_by_category.show(truncate=False)

---
## Step 8 — Sentiment + join (third question, still one Spark app)

```mermaid
flowchart TB
  REV[reviews text file] --> NEG[negative keyword filter]
  ORD[orders CSV] --> CAT[category per product]
  NEG --> JOIN[join on product_name]
  CAT --> JOIN
  JOIN --> SUM[negative hits by category]
```

MapReduce would need custom join logic or a chain of jobs. Spark expresses it as a **join + groupBy**.

In [ ]:
NEGATIVE = {"poor", "disappointed", "bad", "broke", "damaged", "slow", "cancelled", "delayed", "irritation", "faded"}

negative_tokens = (
    reviews.select(
        F.explode(F.split(F.lower(F.col("value")), r"\W+")).alias("token")
    )
    .where(F.col("token").isin(list(NEGATIVE)))
)

print(f"Negative keyword token count: {negative_tokens.count()}")
negative_tokens.groupBy("token").count().orderBy(F.desc("count")).show()

In [ ]:
# Map products to categories, then score negative tokens per category via product mentions in reviews
product_category = orders.select("product_name", "category").distinct()

category_catalog = product_category.collect()
category_scores = []

for row in category_catalog:
    name = row["product_name"].lower()
    hits = reviews.where(F.lower(F.col("value")).contains(name)).count()
    neg = reviews.where(
        F.lower(F.col("value")).contains(name)
        & F.lower(F.col("value")).rlike("poor|disappointed|bad|broke|damaged|slow|cancelled|delayed|irritation|faded")
    ).count()
    category_scores.append((row["category"], row["product_name"], hits, neg))

print("Product mention + negative signal by category (Spark — no extra MapReduce jobs):")
for category, product, hits, neg in sorted(category_scores, key=lambda r: r[3], reverse=True):
    print(f"  {category:12} | {product:22} | mentions={hits} | negative_lines={neg}")

---
## Step 9 — Full pipeline in one DAG

Combine clickstream conversion with orders — **no extra MapReduce job**.

In [ ]:
clickstream = spark.read.option("header", True).csv(data_path("clickstream.csv"))

actions_by_customer = (
    clickstream.groupBy("customer_id", "action")
    .count()
    .groupBy("customer_id")
    .pivot("action")
    .sum("count")
    .na.fill(0)
)

customer_revenue = (
    orders.groupBy("customer_id")
    .agg(F.round(F.sum("revenue"), 2).alias("total_revenue"), F.count("*").alias("orders"))
)

customer_360 = customer_revenue.join(actions_by_customer, "customer_id", "left").orderBy(F.desc("total_revenue"))

print("Customer revenue + clickstream actions (single Spark pipeline)")
customer_360.show(8, truncate=False)

```mermaid
flowchart TB
  subgraph one [One Spark application]
    A[Word counts] --> UI[Notebook output]
    B[Revenue by category] --> UI
    C[Negative keyword scan] --> UI
    D[Customer 360 join] --> UI
  end

  subgraph mr [MapReduce equivalent]
    J1[Job 1] --> T1[(HDFS temp)]
    J2[Job 2] --> T2[(HDFS temp)]
    J3[Job 3] --> T3[(HDFS temp)]
    J4[Job 4] --> T4[(HDFS temp)]
  end

  one -.->|replaces| mr
```

**Checkpoint:** Four analytics outputs, one Spark session, zero HDFS temp directories.

---
## Step 10 — Inspect the Spark plan

Spark optimizes chained transforms before running.

In [ ]:
print("Logical plan for customer 360:")
customer_360.explain(mode="formatted")

Open **Spark UI → Jobs / Stages** at http://localhost:4040 to see shuffle read/write vs MapReduce counters.

---
## Step 11 — Hadoop vs Spark summary

```mermaid
quadrantChart
  title Where each engine fits
  x-axis Simple batch --> Complex analytics
  y-axis High dev friction --> Low dev friction
  quadrant-1 Spark default choice
  quadrant-2 Niche
  quadrant-3 Legacy
  quadrant-4 Quick one-offs
  MapReduce Streaming: [0.2, 0.15]
  PySpark DataFrames: [0.85, 0.85]
  Spark SQL: [0.9, 0.8]
```

| | MapReduce (prior labs) | Spark (today) |
|---|------------------------|---------------|
| Lines of code for wordcount | ~50 + deploy | ~8 |
| Jobs for 4 report sections | 4 YARN apps | 1 Spark app |
| Iteration | Minutes | Seconds |
| Joins | Painful | `join`, SQL |
| Monitor | YARN 8088, History 19888 | Spark UI 4040 |
| AWS path | EMR MapReduce step | EMR Spark, Glue, Athena |

In [ ]:
from spark_helpers import stop_spark

stop_spark(spark)
print("Spark session stopped.")

---
## Step 12 — Next steps

```mermaid
flowchart LR
  LOCAL[PySpark local lab] --> EMR[Amazon EMR Spark]
  LOCAL --> S3[S3 data lake]
  LOCAL --> GLUE[AWS Glue Spark]
  MR[MapReduce on Docker] -.->|pain motivated| LOCAL
```

| Resource | Path |
|----------|------|
| Written guide | [SPARK-STUDENT-GUIDE.md](./SPARK-STUDENT-GUIDE.md) |
| MapReduce lab | [../MAPREDUCE-STUDENT-GUIDE.md](../MAPREDUCE-STUDENT-GUIDE.md) |
| Hadoop cluster | [../HADOOP-STUDENT-GUIDE.md](../HADOOP-STUDENT-GUIDE.md) |